In [17]:
import os
from pathlib import Path

import mne
import torch
import numpy as np
import torch.nn as nn
from torch.nn import functional as F

from eegnet import EEGNet
from train import train, test
from data_loader import load_subject_epochs
from config import TRAIN_DIR, TEST_DIR
from data_loader import load_all
from dataset import build_loaders
from visualize import visualize_layer


/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:

def make_model(F1, D, input_shape, 
               n_classes, classification_type) -> EEGNet:
    _, n_channels, tps = input_shape
    return EEGNet(
        F1, D, n_channels, tps, n_classes, 
        classification_type=classification_type
    )

# Within-subject

In [3]:
# subject = "S001"

# X, y = load_subject_epochs(TRAIN_DIR, subject)

In [ ]:
from dataset import block_kfold_loaders



In [28]:
from collections import Counter

Counter(y)

Counter({np.int64(0): 24, np.int64(1): 21})

In [35]:
Counter(val_dl.dataset.y.numpy())

Counter({np.int64(0): 6, np.int64(1): 5})

In [27]:

X, y = load_subject_epochs(TRAIN_DIR, "S002")

F1, D = 4, 2
input_shape = X.shape
n_classes = 2
classification_type = "within-subject"


fold_results = []
for fold, (train_dl, val_dl, test_dl), meta in block_kfold_loaders(X, y, k=4):
    model = make_model(F1, D, input_shape, 
               n_classes, classification_type)
    train(model, train_dl, val_dl, checkpoint_name=f"S002_fold{fold}.pt")
    acc, auc, _, _ = test(model, test_dl)
    fold_results.append((acc, auc))

print(f"Mean CV accuracy: {np.mean(np.array(fold_results)[:, 0]):.4f}"
      f"Mean ROC-AUC: {np.mean(np.array(fold_results)[:, 1]):.4f}")

  [warn] S002 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S002

Block k-fold CV: k=4, total trials=45
Block sizes: [12, 11, 11, 11]
Assignment per fold: 2 train block(s), 1 val block, 1 test block

  Fold 1/4 — train: 23 trials, val: 11 trials, test: 11 trials

Training on cpu | max_epochs=200 | patience=20
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val AUC    Time
----------------------------------------------------------------------
     1      0.6469     0.7391     0.6911    0.6364    0.7667    0.0s
     2      0.7115     0.4783     0.6907    0.5455    0.7333    0.0s
     3      0.7573     0.3478     0.6902    0.5455    0.7000    0.0s
     4      0.7016     0.4348     0.6898    0.5455    0.7000    0.0s
     5      0.7169     0.3913     0.6893    0.5455    0.7000    0.0s
     6      0.7641     0.3478     0.6889    0.5455    0.7000    0.0s
     7      0.7195     0.5217     0.6884    0.5455    0.7000    0.0s
     8     

/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

     5      0.6414     0.7273     0.6977    0.4167    0.0000    0.0s
     6      0.6761     0.5000     0.6983    0.5000    0.0000    0.0s
     7      0.7083     0.5000     0.6988    0.5000    0.0000    0.0s
     8      0.6748     0.6364     0.6994    0.5000    0.0000    0.0s
     9      0.5847     0.9091     0.7000    0.5000    0.0000    0.0s
    10      0.6447     0.6818     0.7007    0.5000    0.0000    0.0s
    11      0.6284     0.7273     0.7014    0.5000    0.0000    0.0s
    12      0.6804     0.4545     0.7022    0.5000    0.0000    0.0s
    13      0.6432     0.6818     0.7029    0.5000    0.0000    0.0s
    14      0.6572     0.5909     0.7036    0.5000    0.0000    0.0s
    15      0.6849     0.5909     0.7043    0.5000    0.0000    0.0s
    16      0.6602     0.6364     0.7049    0.5000    0.0000    0.1s
    17      0.6890     0.5455     0.7056    0.5000    0.0000    0.1s
    18      0.6472     0.6364     0.7063    0.5000    0.0000    0.0s
    19      0.6321     0.6818     

In [25]:
X_test.shape

(45, 64, 257)

# Cross subjects

In [26]:
# testing model trained on subject S002

from dataset import EEGDataset 
from torch.utils.data import DataLoader

# using other subject as test
X_test, y_test = load_subject_epochs(TEST_DIR, "S009")

test_ds = EEGDataset(X_test, y_test)
# g = torch.Generator().manual_seed(42)

test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)

results = []
acc, auc, _, _ = test(model, test_dl)
results.append(acc)

print(f"Mean Accuracy: {np.mean(results):.4f}")

  [warn] S009 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/test/S009

── Test Results ──────────────────────────────────────
Accuracy : 0.5778  (25/45)
ROC-AUC  : 0.4980

              precision    recall  f1-score   support

  both_fists     0.6429    0.3913    0.4865        23
   both_feet     0.5484    0.7727    0.6415        22

    accuracy                         0.5778        45
   macro avg     0.5956    0.5820    0.5640        45
weighted avg     0.5967    0.5778    0.5623        45

Confusion matrix:
[[ 9 14]
 [ 5 17]]
Mean Accuracy: 0.5778


In [4]:
all_data = load_all(TRAIN_DIR)

Found 8 subjects under data/MNE-eegbci-data/files/eegmmidb/1.0.0/train
  [warn] S001 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S001
  S001: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S002 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S002
  S002: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S003 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S003
  S003: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S004 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S004
  S004: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S005 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/train/S005
  S005: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S006 – no files found for group 'LR

In [5]:
X_train = np.concatenate(
    [data[0] for _, data in all_data.items()]
)
y_train = np.concatenate(
    [data[1] for _, data in all_data.items()]
)

In [6]:
from config import TEST_DIR

all_test_data = load_all(TEST_DIR)
X_test = np.concatenate(
    [data[0] for _, data in all_test_data.items()]
)
y_test = np.concatenate(
    [data[1] for _, data in all_test_data.items()]
)

Found 2 subjects under data/MNE-eegbci-data/files/eegmmidb/1.0.0/test
  [warn] S009 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/test/S009
  S009: 45 trials, X shape (45, 64, 257), classes [0, 1]
  [warn] S010 – no files found for group 'LR' (runs [4, 8, 12]) in data/MNE-eegbci-data/files/eegmmidb/1.0.0/test/S010
  S010: 45 trials, X shape (45, 64, 257), classes [0, 1]


In [7]:
X_train_n, y_train_n = X_train[:-3, :, :], y_train[:-3]
X_val, y_val = X_train[-3:, :, :], y_train[-3:]

In [8]:
train_loader, val_loader, test_loader = build_loaders(X_train_n, y_train_n, X_val, y_val, X_test, y_test)

In [28]:
F1, D = 8, 2
input_shape = X_train.shape
n_classes = 2
classification_type = "cross-subject"


results = []
model = make_model(F1, D, input_shape, 
            n_classes, classification_type)
train(model, train_loader, val_loader, checkpoint_name=f"cross-sub-8-2.pt")



Training on cpu | max_epochs=200 | patience=20
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val AUC    Time
----------------------------------------------------------------------
     1      0.7090     0.4650     0.7016    0.3333    0.0000    6.6s
     2      0.6886     0.5210     0.7052    0.3333    0.0000    4.9s
     3      0.6840     0.5742     0.7045    0.3333    0.5000    7.3s
     4      0.6785     0.5826     0.7002    0.3333    1.0000    7.9s
     5      0.6739     0.6050     0.6853    0.3333    1.0000    7.9s
     6      0.6654     0.5770     0.6549    0.6667    1.0000    6.0s
     7      0.6603     0.6134     0.6322    1.0000    1.0000    5.9s
     8      0.6521     0.6443     0.6075    1.0000    1.0000    6.8s
     9      0.6477     0.6583     0.5873    1.0000    1.0000    6.4s
    10      0.6315     0.6779     0.5771    1.0000    1.0000    5.3s
    11      0.6214     0.7171     0.5737    1.0000    1.0000    7.8s
    12      0.6117     0.6947     0.5745    0.6667   

{'train_loss': [0.7089726874808303,
  0.6885663367786995,
  0.6839996918576772,
  0.6785049199724064,
  0.6739475702037331,
  0.6653959856313818,
  0.660268129086962,
  0.6520690036421063,
  0.6477358152552479,
  0.631525616185004,
  0.6214113517635677,
  0.6116663496367404,
  0.6194107964927075,
  0.5897969002483272,
  0.5881704779900089,
  0.5807901632552054,
  0.579683542919426,
  0.5807520540154615,
  0.5586526819637844,
  0.5393126725482673,
  0.5479278228863949,
  0.5418209390146058,
  0.5379412898830339,
  0.5306043412838998,
  0.4897948219662621,
  0.4873835021207313,
  0.5051706045114693,
  0.48710087242246675,
  0.48517910489181176,
  0.4697475873288654,
  0.44554397856154027,
  0.44023455474890916,
  0.43649881460419554,
  0.4357671896282698,
  0.426290096855965,
  0.39056814067503987,
  0.40728050887751643,
  0.3968257676152622,
  0.4044216287737133,
  0.40173321394693284,
  0.355651509194147,
  0.35560566537520466,
  0.35958787598529784,
  0.31190949427981335,
  0.35339889

In [29]:

acc, auc, _, _ = test(model, test_loader)
results.append(acc)

print(f"Mean Accuracy: {np.mean(results):.4f}")


── Test Results ──────────────────────────────────────
Accuracy : 0.6222  (56/90)
ROC-AUC  : 0.6774

              precision    recall  f1-score   support

  both_fists     0.6111    0.7174    0.6600        46
   both_feet     0.6389    0.5227    0.5750        44

    accuracy                         0.6222        90
   macro avg     0.6250    0.6201    0.6175        90
weighted avg     0.6247    0.6222    0.6184        90

Confusion matrix:
[[33 13]
 [21 23]]
Mean Accuracy: 0.6222


In [16]:
# Test accuracy as well as test roc-auc is significantly lower
# than train and val

# Reducing the datapoints further


In [47]:
X_train.max(), X_train.min()

(np.float64(0.00046528067890147984), np.float64(-0.00041759501674098483))

In [31]:
X_train_n, y_train_n = X_train[:-5, :, :], y_train[:-5]
X_val, y_val = X_train[-2:, :, :], y_train[-2:]

train_loader, val_loader, test_loader = build_loaders(X_train_n, y_train_n, X_val, y_val, X_test, y_test)


F1, D = 4, 2
input_shape = X_train.shape
n_classes = 2
classification_type = "cross-subject"


results = []
model = make_model(F1, D, input_shape, 
            n_classes, classification_type)
train(model, train_loader, val_loader, checkpoint_name=f"cross-sub-reduced-vars-8-2.pt")


acc, auc, _, _ = test(model, test_loader)
results.append(acc)

print(f"Mean Accuracy: {np.mean(results):.4f}")


Training on cpu | max_epochs=200 | patience=20
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val AUC    Time
----------------------------------------------------------------------
     1      0.6916     0.5324     0.6964    0.5000    0.0000    4.7s
     2      0.6941     0.5070     0.6991    0.5000    0.0000    6.4s
     3      0.6882     0.5493     0.7023    0.5000    0.0000    6.3s
     4      0.6885     0.5577     0.7060    0.5000    0.0000    5.3s
     5      0.6839     0.5634     0.7122    0.5000    0.0000    4.8s
     6      0.6758     0.5521     0.7157    0.5000    0.0000    5.0s
     7      0.6815     0.5606     0.7150    0.5000    0.0000    5.6s
     8      0.6732     0.5803     0.7128    0.5000    0.0000    5.4s
     9      0.6691     0.5887     0.7163    0.5000    0.0000    5.8s
    10      0.6695     0.6254     0.7248    0.5000    0.0000    4.4s
    11      0.6674     0.5915     0.7314    0.5000    0.0000    4.8s
    12      0.6620     0.6169     0.7348    0.5000   

In [ ]:
# when we further reduce the datapoints the performance
# deteriorates further


/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Visualization

In [46]:
for m in model.modules():
    print(m)

EEGNet(
  (block1): EEGNetBlock1(
    (f1_conv2d): Conv2d(1, 4, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (batchnorm1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (depthwise_spatial_conv2d): Conv2d(4, 8, kernel_size=(64, 1), stride=(1, 1), groups=4, bias=False)
    (batchnorm2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (avg_pool2d): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (dropout): Dropout(p=0.25, inplace=False)
  )
  (block2): EEGNetBlock2(
    (_sep_conv2d_temporal): Conv2d(8, 8, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=8, bias=False)
    (_sep_conv2d_pointwise): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (batchnorm): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (avg_pool2d): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (dropout): Drop

In [19]:
# Spatial filter output — separability between classes
fig, features, labels = visualize_layer(model, test_loader, layer_name="block1.f1_conv2d")

# # After temporal separable conv — full learned representation
# visualize_layer(model, test_loader, layer_name="separable_conv")

# # Penultimate dense layer — what the classifier actually sees
# visualize_layer(model, test_loader, layer_name="fc")


── Feature visualisation: layer='block1.f1_conv2d' ──────────────


/media/ashatya/Data3/eegnet_study/visualize.py:143: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  [torch.tensor(f) for f in hook.features], dim=0


[extract_features] layer='block1.f1_conv2d'  raw shape: (90, 66048)  labels: (90,)
  Running t-SNE  (perplexity=30.0, n_iter=1000) ...
  Running UMAP  (n_neighbors=15, min_dist=0.1) ...


/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [20]:
import matplotlib.pyplot as plt

In [23]:
features.shape

(90, 66048)

In [27]:
from visualize import visualize_layer, print_layers

# Confirm names on your actual model instance
print_layers(model)

# Recommended — final feature vector before classifier
visualize_layer(
    model, test_loader,
    layer_name="block2.flatten",
    title="EEGNet block2.flatten — test set",
    save_path="figures/block2_flatten.png",
)

# Compare early vs late representations
for layer in ["block1.batchnorm2", "block2.batchnorm", "block2.flatten"]:
    visualize_layer(
        model, test_loader,
        layer_name=layer,
        save_path=f"figures/{layer.replace('.', '_')}.png",
    )


Available layers in model:
  block1                                   EEGNetBlock1
  block1.f1_conv2d                         Conv2d
  block1.batchnorm1                        BatchNorm2d
  block1.depthwise_spatial_conv2d          Conv2d
  block1.batchnorm2                        BatchNorm2d
  block1.avg_pool2d                        AvgPool2d
  block1.dropout                           Dropout
  block2                                   EEGNetBlock2
  block2._sep_conv2d_temporal              Conv2d
  block2._sep_conv2d_pointwise             Conv2d
  block2.batchnorm                         BatchNorm2d
  block2.avg_pool2d                        AvgPool2d
  block2.dropout                           Dropout
  block2.flatten                           Flatten
  block2.dense                             Linear

── Feature visualisation: layer='block2.flatten' ──────────────


/media/ashatya/Data3/eegnet_study/visualize.py:143: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  [torch.tensor(f) for f in hook.features], dim=0


[extract_features] layer='block2.flatten'  raw shape: (90, 64)  labels: (90,)
  Running t-SNE  (perplexity=30.0, n_iter=1000) ...
  Running UMAP  (n_neighbors=15, min_dist=0.1) ...


/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Saved combined plot → figures/block2_flatten.png

── Feature visualisation: layer='block1.batchnorm2' ──────────────


...

[extract_features] layer='block2.batchnorm'  raw shape: (90, 512)  labels: (90,)
  Running t-SNE  (perplexity=30.0, n_iter=1000) ...
  Running UMAP  (n_neighbors=15, min_dist=0.1) ...
  Saved combined plot → figures/block2_batchnorm.png

── Feature visualisation: layer='block2.flatten' ──────────────


/media/ashatya/Data3/eegnet_study/visualize.py:143: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  [torch.tensor(f) for f in hook.features], dim=0


[extract_features] layer='block2.flatten'  raw shape: (90, 64)  labels: (90,)
  Running t-SNE  (perplexity=30.0, n_iter=1000) ...
  Running UMAP  (n_neighbors=15, min_dist=0.1) ...


/media/ashatya/Data3/eegnet_study/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Saved combined plot → figures/block2_flatten.png


# Improvement

## 1. Dropout  = 0.5 for all exp

In [54]:
from typing import Tuple, Literal

import torch
import torch.nn as nn
from torch.nn import functional as F

__all__ = ["EEGNet"]

class EEGNetBlock1(nn.Module):
    def __init__(self, 
                 f1: int, 
                 depth_mul: int, 
                 n_channels: int,
                 drop: float=0.5):
        super().__init__()
        self.f1_conv2d = nn.Conv2d(kernel_size=(1, 64), # acc. to paper
                                   in_channels=1,
                                   out_channels=f1, # acc. to paper
                                   padding=(0, 32),  # to match author's keras code 
                                   bias=False
                                   ) 
        
        
        self.batchnorm1 = nn.BatchNorm2d(f1)
        
        self.depthwise_spatial_conv2d = nn.Conv2d(in_channels=f1, 
                                                  out_channels=f1*depth_mul,    # multiplied by depth
                                                  kernel_size=(n_channels, 1),
                                                  groups=f1,
                                                  bias=False
                                                  )
        
        self.batchnorm2 = nn.BatchNorm2d(f1*depth_mul)
        
        self.avg_pool2d = nn.AvgPool2d((1, 4))  # acc. to paper
        
        self.dropout = nn.Dropout(p=drop)    # acc. to paper
        
    def forward(self, x: torch.Tensor):
        
        # 1. reshape
        # x = x.unsqueeze(1)  # (1, C, T)
        
        # 2. Linear activation(conv2d)
        x = self.f1_conv2d(x)   # (f1, C, T)
        
        # 3. batchnorm
        x = self.batchnorm1(x)   # (f1, C, T)
        
        # 4. depthwise conv2d
        x = self.depthwise_spatial_conv2d(x)    # (f1*depth_mul, 1, T)
        
        # 5. batchnorm
        x = self.batchnorm2(x)  # (f1*depth_mul, 1, T) 
        
        # 6. elu activation
        x = F.elu(x)
        
        # 7. avg pooling
        x = self.avg_pool2d(x)  # (f1*depth_mul, 1, T//4)
        
        # 8. dropout
        x = self.dropout(x)     # (f1*depth_mul, 1, T//4)
        
        return x
        
        
class EEGNetBlock2(nn.Module):
    def __init__(self, 
                 n_classes: int,
                 input_shape: Tuple,
                 f2: int,   # first separable conv 2d out channels 
                 drop: float=0.5):
        super().__init__()
        channels, _, tps = input_shape
        self._sep_conv2d_temporal = nn.Conv2d(in_channels=channels,
                                            out_channels=f2,
                                            kernel_size=(1, 16),
                                            groups=channels,
                                            bias=False,
                                            padding="same") # to retain shape
        
        self._sep_conv2d_pointwise = nn.Conv2d(in_channels=channels,
                                              out_channels=f2,
                                              kernel_size=1, 
                                              bias=False)
        
        self.batchnorm = nn.BatchNorm2d(f2)

        self.avg_pool2d = nn.AvgPool2d((1, 8))  # acc. paper
        
        self.dropout = nn.Dropout(p=drop)
        
        self.flatten = nn.Flatten()
        
        # f2*(tps // 8)
        self.dense = nn.Linear(in_features=4032, out_features=n_classes, bias=False)

    def depthwise_separable(self, x: torch.Tensor):
        x = self._sep_conv2d_temporal(x)    # 
        x = self._sep_conv2d_pointwise(x)
        return x

    def forward(self, x: torch.Tensor):
        # 1. separable conv
        x = self.depthwise_separable(x) # 
        
        # 2. batchnorm
        x = self.batchnorm(x)
        
        # 3. activation
        x = F.elu(x)
        
        # 4. avg pool
        x = self.avg_pool2d(x)
        
        # 5. dropout
        x = self.dropout(x)
        
        # 6. flatten
        x = self.flatten(x)
        
        # 7. dense
        x = self.dense(x)
        
        return x
        
class NewEEGNet(nn.Module):
    """
    Implemented Paper
    
    For training - 
    Vary: f1, D
    
    `f2 = D * f1`  
    But, in principle it can take any value. 
    `f2 < (>) D * f1` denotes a compressed (overcomplete) representation, 
    learning a fewer (more) feature maps than inputs.
    
    """
    def __init__(self, 
                 f1: int, 
                 depth_mul: int, 
                 n_channels: int,   # C or dim 0
                 tps: int,  # T or time points or dim 1
                 n_classes: int,
                 f2: int=None,
                 classification_type: Literal["within-subject", "cross-subject"] = "within-subject"
                 ):
        super().__init__()
        self.block1 = EEGNetBlock1(f1, 
                                   depth_mul, 
                                   n_channels, 
                                   0.5)     # dropout = 0.5 for both cases
        self.block2 = EEGNetBlock2(n_classes, 
                                   (depth_mul * f1, 1, tps // 4),
                                   f2 if f2 else f1*depth_mul,    # f2, acc. to paper
                                   0.5)     # dropout = 0.5 for both cases
        
    def forward(self, x: torch.Tensor):
        # 1. converting to volts from micro volts
        x = x * 1e6
        
        x = self.block1(x)
        x = self.block2(x)
        
        return x
        


In [33]:
from new_eegnet import EEGNet as NewEEGNet

In [56]:
_, n_channels, tps = input_shape
F1, D = 4, 2
model = NewEEGNet(F1, D, n_classes, tps, n_classes, classification_type=classification_type)

In [50]:
X_train_n.shape

(355, 64, 257)

In [51]:
dummy = torch.randn((1, 1, 64, 257))

In [57]:
y_dummy = model(dummy)
y_dummy.shape

torch.Size([1, 2])

In [58]:
y_dummy

tensor([[-0.1029,  0.6223]], grad_fn=<MmBackward0>)

In [59]:

results = []
train(model, train_loader, val_loader, checkpoint_name=f"drop-20perc-cross-sub-{F1}-{D}.pt")



Training on cpu | max_epochs=200 | patience=20
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val AUC    Time
----------------------------------------------------------------------
     1      0.7040     0.5211     0.6888    0.5000    1.0000   27.3s
     2      0.7025     0.5380     0.6949    0.5000    0.0000   35.9s
     3      0.6674     0.5775     0.6889    0.5000    1.0000   22.1s
     4      0.6493     0.6451     0.6702    0.5000    1.0000   24.9s
     5      0.6413     0.6535     0.6184    1.0000    1.0000   40.5s
     6      0.6168     0.6845     0.5893    0.5000    1.0000   39.4s
     7      0.5940     0.7296     0.5429    1.0000    1.0000   30.1s
     8      0.5963     0.6789     0.4989    1.0000    1.0000   27.2s
     9      0.5639     0.7352     0.4741    1.0000    1.0000   20.9s
    10      0.5287     0.7746     0.4339    1.0000    1.0000   14.8s
    11      0.5212     0.7634     0.4135    1.0000    1.0000   30.7s
    12      0.5085     0.7437     0.4002    1.0000   

KeyboardInterrupt: 

## 2. Convolution kernel size (1, 16) or (1, 32) in first layer of first block

In [ ]:
from typing import Tuple, Literal

import torch
import torch.nn as nn
from torch.nn import functional as F

__all__ = ["EEGNet"]

class EEGNetBlock1(nn.Module):
    def __init__(self, 
                 f1: int, 
                 depth_mul: int, 
                 n_channels: int,
                 drop: float=0.5):
        super().__init__()
        self.f1_conv2d = nn.Conv2d(kernel_size=(1, 16),
                                   in_channels=1,
                                   out_channels=f1, # acc. to paper
                                   padding="same",  # to match author's keras code 
                                   bias=False
                                   ) 
        
        
        self.batchnorm1 = nn.BatchNorm2d(f1)
        
        self.depthwise_spatial_conv2d = nn.Conv2d(in_channels=f1, 
                                                  out_channels=f1*depth_mul,    # multiplied by depth
                                                  kernel_size=(n_channels, 1),
                                                  groups=f1,
                                                  bias=False
                                                  )
        
        self.batchnorm2 = nn.BatchNorm2d(f1*depth_mul)
        
        self.avg_pool2d = nn.AvgPool2d((1, 4))  # acc. to paper
        
        self.dropout = nn.Dropout(p=drop)    # acc. to paper
        
    def forward(self, x: torch.Tensor):
        
        # 1. reshape
        # x = x.unsqueeze(1)  # (1, C, T)
        
        # 2. Linear activation(conv2d)
        x = self.f1_conv2d(x)   # (f1, C, T)
        
        # 3. batchnorm
        x = self.batchnorm1(x)   # (f1, C, T)
        
        # 4. depthwise conv2d
        x = self.depthwise_spatial_conv2d(x)    # (f1*depth_mul, 1, T)
        
        # 5. batchnorm
        x = self.batchnorm2(x)  # (f1*depth_mul, 1, T) 
        
        # 6. elu activation
        x = F.elu(x)
        
        # 7. avg pooling
        x = self.avg_pool2d(x)  # (f1*depth_mul, 1, T//4)
        
        # 8. dropout
        x = self.dropout(x)     # (f1*depth_mul, 1, T//4)
        
        return x
        
        
class EEGNetBlock2(nn.Module):
    def __init__(self, 
                 n_classes: int,
                 input_shape: Tuple,
                 f2: int,   # first separable conv 2d out channels 
                 drop: float=0.5):
        super().__init__()
        channels, _, tps = input_shape
        self._sep_conv2d_temporal = nn.Conv2d(in_channels=channels,
                                            out_channels=f2,
                                            kernel_size=(1, 16),
                                            groups=channels,
                                            bias=False,
                                            padding="same") # to retain shape
        
        self._sep_conv2d_pointwise = nn.Conv2d(in_channels=channels,
                                              out_channels=f2,
                                              kernel_size=1, 
                                              bias=False)
        
        self.batchnorm = nn.BatchNorm2d(f2)

        self.avg_pool2d = nn.AvgPool2d((1, 8))  # acc. paper
        
        self.dropout = nn.Dropout(p=drop)
        
        self.flatten = nn.Flatten()
        
        # f2*(tps // 8)
        self.dense = nn.Linear(in_features=4032, out_features=n_classes, bias=False)

    def depthwise_separable(self, x: torch.Tensor):
        x = self._sep_conv2d_temporal(x)    # 
        x = self._sep_conv2d_pointwise(x)
        return x

    def forward(self, x: torch.Tensor):
        # 1. separable conv
        x = self.depthwise_separable(x) # 
        
        # 2. batchnorm
        x = self.batchnorm(x)
        
        # 3. activation
        x = F.elu(x)
        
        # 4. avg pool
        x = self.avg_pool2d(x)
        
        # 5. dropout
        x = self.dropout(x)
        
        # 6. flatten
        x = self.flatten(x)
        
        # 7. dense
        x = self.dense(x)
        
        return x
        
class NewEEGNet(nn.Module):
    """
    Implemented Paper
    
    For training - 
    Vary: f1, D
    
    `f2 = D * f1`  
    But, in principle it can take any value. 
    `f2 < (>) D * f1` denotes a compressed (overcomplete) representation, 
    learning a fewer (more) feature maps than inputs.
    
    """
    def __init__(self, 
                 f1: int, 
                 depth_mul: int, 
                 n_channels: int,   # C or dim 0
                 tps: int,  # T or time points or dim 1
                 n_classes: int,
                 f2: int=None,
                 classification_type: Literal["within-subject", "cross-subject"] = "within-subject"
                 ):
        super().__init__()
        self.block1 = EEGNetBlock1(f1, 
                                   depth_mul, 
                                   n_channels, 
                                   0.5)     # dropout = 0.5 for both cases
        self.block2 = EEGNetBlock2(n_classes, 
                                   (depth_mul * f1, 1, tps // 4),
                                   f2 if f2 else f1*depth_mul,    # f2, acc. to paper
                                   0.5)     # dropout = 0.5 for both cases
        
    def forward(self, x: torch.Tensor):
        # 1. converting to volts from micro volts
        x = x * 1e6
        
        x = self.block1(x)
        x = self.block2(x)
        
        return x
        


## Adding residual

In [61]:
from typing import Tuple, Literal

import torch
import torch.nn as nn
from torch.nn import functional as F

__all__ = ["EEGNet"]


class ResEEGNetBlock1(nn.Module):
    def __init__(self,
                 f1: int,
                 depth_mul: int,
                 n_channels: int,
                 drop: float = 0.5):
        super().__init__()
        self.f1_conv2d = nn.Conv2d(kernel_size=(1, 16),
                                   in_channels=1,
                                   out_channels=f1,
                                   padding="same",
                                   bias=False)

        self.batchnorm1 = nn.BatchNorm2d(f1)

        self.depthwise_spatial_conv2d = nn.Conv2d(in_channels=f1,
                                                  out_channels=f1 * depth_mul,
                                                  kernel_size=(n_channels, 1),
                                                  groups=f1,
                                                  bias=False)

        self.batchnorm2 = nn.BatchNorm2d(f1 * depth_mul)

        self.avg_pool2d = nn.AvgPool2d((1, 4))

        self.dropout = nn.Dropout(p=drop)

        # Shortcut: match (1, C, T) -> (f1*depth_mul, 1, T//4)
        # 1x1 conv handles channel change; avg_pool matches the T//4 reduction
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels=1,
                      out_channels=f1 * depth_mul,
                      kernel_size=(n_channels, 1),   # collapses spatial dim C -> 1
                      bias=False),
            nn.BatchNorm2d(f1 * depth_mul),
            nn.AvgPool2d((1, 4)),
        )

    def forward(self, x: torch.Tensor):
        residual = self.shortcut(x)

        x = self.f1_conv2d(x)
        x = self.batchnorm1(x)
        x = self.depthwise_spatial_conv2d(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.avg_pool2d(x)
        x = self.dropout(x)

        x = x + residual
        return x


class ResEEGNetBlock2(nn.Module):
    def __init__(self,
                 n_classes: int,
                 input_shape: Tuple,
                 f2: int,
                 drop: float = 0.5):
        super().__init__()
        channels, _, tps = input_shape

        self._sep_conv2d_temporal = nn.Conv2d(in_channels=channels,
                                              out_channels=f2,
                                              kernel_size=(1, 16),
                                              groups=channels,
                                              bias=False,
                                              padding="same")

        self._sep_conv2d_pointwise = nn.Conv2d(in_channels=channels,
                                               out_channels=f2,
                                               kernel_size=1,
                                               bias=False)

        self.batchnorm = nn.BatchNorm2d(f2)

        self.avg_pool2d = nn.AvgPool2d((1, 8))

        self.dropout = nn.Dropout(p=drop)

        self.flatten = nn.Flatten()

        self.dense = nn.Linear(in_features=4032, out_features=n_classes, bias=False)

        # Shortcut: match (channels, 1, tps) -> (f2, 1, tps//8)
        # 1x1 conv handles channel change; avg_pool matches the T//8 reduction
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels=channels,
                      out_channels=f2,
                      kernel_size=1,
                      bias=False),
            nn.BatchNorm2d(f2),
            nn.AvgPool2d((1, 8)),
        )

    def depthwise_separable(self, x: torch.Tensor):
        x = self._sep_conv2d_temporal(x)
        x = self._sep_conv2d_pointwise(x)
        return x

    def forward(self, x: torch.Tensor):
        residual = self.shortcut(x)

        x = self.depthwise_separable(x)
        x = self.batchnorm(x)
        x = F.elu(x)
        x = self.avg_pool2d(x)
        x = self.dropout(x)

        x = x + residual

        x = self.flatten(x)
        x = self.dense(x)
        return x


class ResEEGNet(nn.Module):
    def __init__(self,
                 f1: int,
                 depth_mul: int,
                 n_channels: int,
                 tps: int,
                 n_classes: int,
                 f2: int = None,
                 classification_type: Literal["within-subject", "cross-subject"] = "within-subject"
                 ):
        super().__init__()
        self.block1 = ResEEGNetBlock1(f1, depth_mul, n_channels, drop=0.5)
        self.block2 = ResEEGNetBlock2(n_classes,
                                   (depth_mul * f1, 1, tps // 4),
                                   f2 if f2 else f1 * depth_mul,
                                   drop=0.5)

    def forward(self, x: torch.Tensor):
        x = self.block1(x)
        x = self.block2(x)
        return x

In [62]:
_, n_channels, tps = input_shape
F1, D = 4, 2
model = ResEEGNet(F1, D, n_classes, tps, n_classes, classification_type=classification_type)

In [63]:

results = []
train(model, train_loader, val_loader, checkpoint_name=f"res-cross-sub-{F1}-{D}.pt")



Training on cpu | max_epochs=200 | patience=20
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val AUC    Time
----------------------------------------------------------------------
     1      0.6897     0.5634     0.6711    0.5000    1.0000   34.0s
     2      0.6530     0.6197     0.6682    0.5000    1.0000   20.3s
     3      0.5987     0.6789     0.6268    0.5000    1.0000   44.6s
     4      0.5930     0.6873     0.5745    0.5000    1.0000   31.5s
     5      0.5556     0.7211     0.5340    0.5000    1.0000   24.2s
     6      0.5279     0.7465     0.4797    0.5000    1.0000   27.4s
     7      0.5195     0.7606     0.4684    0.5000    1.0000   35.7s


KeyboardInterrupt: 